In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import cv2

Important notions applied:<br>
1. Sorting the directories to ensure each mri[i] corresponds to mask[i]<br>
2. MRI's are in .jpg format so they are opened using all 3 channels, masks are binary so they are opened using only 1, grayscale <br>
3. All images are resized to 256x256 and then turned into tensors by modifying their type into 'float32'<br>

In [ ]:
images=[]
masks=[]
image_path="Dataset_2/images"
mask_path="Dataset_2/masks"
image_files= sorted(os.listdir(image_path)) #important 
mask_files= sorted(os.listdir(mask_path)) 

In [ ]:
for filename in image_files:
    path=os.path.join(image_path, filename)
    img=cv2.imread(path)
    img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img=cv2.resize(img, (256, 256))
    img=img.astype('float32')/255.0
    images.append(img)

In [ ]:
for filename in mask_files:
    path=os.path.join(mask_path, filename)
    mask=cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask=cv2.resize(mask, (256, 256))
    mask=(mask > 127).astype('float32')
    mask=np.expand_dims(mask, axis=-1)
    masks.append(mask)

In [ ]:
X=np.array(images)
y=np.array(masks)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val= train_test_split(X_train,y_train,test_size=0.15, random_state=42)

# UNet


In [ ]:
import tensorflow as tf
def dice_score(y_true, y_pred, smooth=1):
   y_true_f= tf.keras.backend.flatten(y_true)
   y_pred_f= tf.keras.backend.flatten(y_pred)
   intersection= tf.keras.backend.sum(y_true_f * y_pred_f)
   return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

In [ ]:
def unet(input_shape=(256,256,3)):
    inputs=tf.keras.layers.Input(input_shape)
    c1=tf.keras.layers.Conv2D(64,3,activation='relu',padding='same')(inputs)
    c1=tf.keras.layers.Conv2D(64,3,activation='relu',padding='same')(c1)

    p1=tf.keras.layers.MaxPooling2D((2,2))(c1)
    
    c2=tf.keras.layers.Conv2D(128,3,activation='relu',padding='same')(p1)
    c2=tf.keras.layers.Conv2D(128,3,activation='relu',padding='same')(c2)

    p2=tf.keras.layers.MaxPooling2D((2,2))(c2)

    c3=tf.keras.layers.Conv2D(256,3,activation='relu',padding='same')(p2)
    c3=tf.keras.layers.Conv2D(256,3,activation='relu',padding='same')(c3)

    p3=tf.keras.layers.MaxPooling2D((2,2))(c3)

    c4=tf.keras.layers.Conv2D(512,3,activation='relu',padding='same')(p3)
    c4=tf.keras.layers.Conv2D(512,3,activation='relu',padding='same')(c4)
    d4=tf.keras.layers.Dropout(0.5)(c4)
    p4=tf.keras.layers.MaxPooling2D((2,2))(d4)

    c5=tf.keras.layers.Conv2D(1024,3,activation='relu',padding='same')(p4)
    c5=tf.keras.layers.Conv2D(1024,3,activation='relu',padding='same')(c5)
    d5=tf.keras.layers.Dropout(0.5)(c5)
    
    u6=tf.keras.layers.Conv2D(512,2,activation='relu',padding='same')(tf.keras.layers.UpSampling2D((2,2))(d5))
    merge6=tf.keras.layers.concatenate([c4,u6],axis=3)

    c6=tf.keras.layers.Conv2D(512,3,activation='relu',padding='same')(merge6)
    c6=tf.keras.layers.Conv2D(512,3,activation='relu',padding='same')(c6)
    u7=tf.keras.layers.Conv2D(256,2,activation='relu',padding='same')(tf.keras.layers.UpSampling2D((2,2))(c6))
    merge7=tf.keras.layers.concatenate([c3,u7],axis=3)
    
    c7=tf.keras.layers.Conv2D(256,3,activation='relu',padding='same')(merge7)
    c7=tf.keras.layers.Conv2D(256,3,activation='relu',padding='same')(c7)
    u8=tf.keras.layers.Conv2D(128,2,activation='relu',padding='same')(tf.keras.layers.UpSampling2D((2,2))(c7))
    merge8=tf.keras.layers.concatenate([c2,u8],axis=3)

    c8=tf.keras.layers.Conv2D(128,3,activation='relu',padding='same')(merge8)
    c8=tf.keras.layers.Conv2D(128,3,activation='relu',padding='same')(c8)
    u9=tf.keras.layers.Conv2D(64,2,activation='relu',padding='same')(tf.keras.layers.UpSampling2D((2,2))(c8))
    merge9=tf.keras.layers.concatenate([c1,u9],axis=3)

    c9=tf.keras.layers.Conv2D(64,3,activation='relu',padding='same')(merge9)
    c9=tf.keras.layers.Conv2D(64,3,activation='relu',padding='same')(c9)
    c9=tf.keras.layers.Conv2D(2,3,activation='relu',padding='same')(c9)
    outputs=tf.keras.layers.Conv2D(1,1,activation='sigmoid')(c9)

    model=tf.keras.Model(inputs=inputs,outputs=outputs)
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=[dice_score])
    #we can also try with loss=tf.keras.losses.Dice()  
    return model



In [ ]:
model=unet()
model.fit(X_train,y_train,
batch_size=4,
epochs=3)